# Model Training — PMS Task Overdue Prediction

Two independent tracks:
1. **Creation-time** (features known at task assignment)
2. **Halfway** (creation + accumulation features)

Each gets its own Optuna tuning, fold-safe CV, and final model.
Split: **70% train / 15% val / 15% test**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from copy import deepcopy
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, precision_recall_curve, auc
)
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
import optuna

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

RANDOM_STATE = 42
N_TRIALS = 50
print('Libraries loaded.')

---
## 1. Load Datasets

In [ ]:
DATA_DIR = Path('../../data/v1')

df_creation = pd.read_csv(DATA_DIR / 'dataset_at_creation_fixed_end_date.csv')
df_halfway  = pd.read_csv(DATA_DIR / 'dataset_at_halfway_fixed_end_date.csv')

assert df_creation['id'].equals(df_halfway['id']), 'IDs must match'
assert df_creation['calculated_overdue'].equals(df_halfway['calculated_overdue']), 'Targets must match'

target = 'calculated_overdue'
id_col = 'id'
y = df_creation[target]

print(f'Creation: {df_creation.shape[0]} rows, {df_creation.shape[1]} cols')
print(f'Halfway:  {df_halfway.shape[0]} rows, {df_halfway.shape[1]} cols')

## 2. Feature Selection & Preprocessing

In [ ]:
DROPPED_BOTH = ['ma_comment_count', 'wl_low', 'kpi_comment_count']
DROPPED_HALFWAY = DROPPED_BOTH + ['subtask_completion_pct', 'subtask_overdue_rate']

Xc = df_creation.drop(columns=[id_col, target])
Xh = df_halfway.drop(columns=[id_col, target])

Xc = Xc.drop(columns=[c for c in DROPPED_BOTH if c in Xc.columns])
Xh = Xh.drop(columns=[c for c in DROPPED_HALFWAY if c in Xh.columns])

for df in [Xc, Xh]:
    if 'planned_duration' in df.columns:
        df['planned_duration'] = df['planned_duration'].clip(lower=0)
    if 'num_ma_revisions' in df.columns:
        df['num_ma_revisions'] = df['num_ma_revisions'].clip(upper=78)
    if 'num_revisions' in df.columns:
        df['num_revisions'] = df['num_revisions'].clip(upper=9)

challenge_flags = ['has_challenges', 'has_kpi_challenge', 'has_kpi_potential_challenge', 'has_subtask_challenge']
existing_c = [c for c in challenge_flags if c in Xc.columns]
existing_h = [c for c in challenge_flags if c in Xh.columns]
if existing_c:
    Xc['total_challenge_load'] = Xc[existing_c].sum(axis=1)
if existing_h:
    Xh['total_challenge_load'] = Xh[existing_h].sum(axis=1)

LEAKY_FEATURES = ['position_id_encoded', 'dept_past_overdue_rate', 'dept_avg_revisions',
                  'emp_past_overdue_rate', 'pos_past_overdue_rate']

print(f'Creation: {Xc.shape[1]} features')
print(f'Halfway:  {Xh.shape[1]} features')

## 3. Train / Val / Test Split (70 / 15 / 15)

Same indices used for both variants.

In [ ]:
Xc_train, Xc_temp, Xh_train, Xh_temp, y_train, y_temp = train_test_split(
    Xc, Xh, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)
Xc_val, Xc_test, Xh_val, Xh_test, y_val, y_test = train_test_split(
    Xc_temp, Xh_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

# Fold-safe variants (drop leaky features)
leaky_in_c = [c for c in LEAKY_FEATURES if c in Xc_train.columns]
leaky_in_h = [c for c in LEAKY_FEATURES if c in Xh_train.columns]

Xc_train_fs = Xc_train.drop(columns=leaky_in_c)
Xc_val_fs   = Xc_val.drop(columns=leaky_in_c)
Xc_test_fs  = Xc_test.drop(columns=leaky_in_c)
Xh_train_fs = Xh_train.drop(columns=leaky_in_h)
Xh_val_fs   = Xh_val.drop(columns=leaky_in_h)
Xh_test_fs  = Xh_test.drop(columns=leaky_in_h)

print(f'Train: {Xh_train.shape[0]}  Val: {Xh_val.shape[0]}  Test: {Xh_test.shape[0]}')

---
## 4. Helper: evaluate & Optuna objective

In [ ]:
def evaluate_model(model, X_train, X_val, y_train, y_val, name='Model'):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_proba = model.predict_proba(X_val)[:, 1]
    return {
        'model': name,
        'accuracy': accuracy_score(y_val, y_pred),
        'precision': precision_score(y_val, y_pred, zero_division=0),
        'recall': recall_score(y_val, y_pred, zero_division=0),
        'f1': f1_score(y_val, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_val, y_proba),
    }

def make_objective(model_name, X, y):
    def objective(trial):
        if model_name == 'XGBoost':
            p = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
                'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
                'gamma': trial.suggest_float('gamma', 0, 5),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
                'random_state': RANDOM_STATE,
                'eval_metric': 'logloss',
            }
            model = XGBClassifier(**p)
        elif model_name == 'CatBoost':
            p = {
                'iterations': trial.suggest_int('iterations', 100, 500, step=50),
                'depth': trial.suggest_int('depth', 4, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
                'border_count': trial.suggest_int('border_count', 32, 255),
                'random_seed': RANDOM_STATE,
                'verbose': 0,
            }
            model = CatBoostClassifier(**p)
        else:
            p = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
                'max_depth': trial.suggest_int('max_depth', 3, 12),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 15, 127),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
                'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
                'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
                'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
                'random_state': RANDOM_STATE,
                'verbose': 0,
            }
            model = LGBMClassifier(**p)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        return cross_val_score(model, X, y, cv=cv, scoring='roc_auc').mean()
    return objective

def tune_one(model_name, X, y):
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
    study.optimize(make_objective(model_name, X, y), n_trials=N_TRIALS, show_progress_bar=True)
    bp = study.best_params
    fixed = {'random_state': RANDOM_STATE, 'verbose': 0}
    if model_name == 'XGBoost':
        fixed['eval_metric'] = 'logloss'
    if model_name == 'CatBoost':
        fixed['random_seed'] = RANDOM_STATE; fixed.pop('random_state')
    mc = {'XGBoost': XGBClassifier, 'CatBoost': CatBoostClassifier, 'LightGBM': LGBMClassifier}[model_name]
    return mc(**{**bp, **fixed}), study.best_value, bp

def train_final(best_model, best_name, X_tr, X_v, y_tr, y_v):
    X_full = pd.concat([X_tr, X_v], axis=0)
    y_full = pd.concat([y_tr, y_v], axis=0)
    bp = best_model.get_params()
    if best_name == 'XGBoost':
        bp['eval_metric'] = 'logloss'
        bp['early_stopping_rounds'] = 20
        m = XGBClassifier(**bp)
        m.fit(X_full, y_full, eval_set=[(X_tr, y_tr), (X_v, y_v)], verbose=0)
    elif best_name == 'CatBoost':
        bp.pop('verbose', None)
        m = CatBoostClassifier(**bp, verbose=0, eval_metric='Logloss')
        m.fit(X_full, y_full, eval_set=(X_v, y_v))
    else:
        bp['early_stopping_rounds'] = 20
        m = LGBMClassifier(**bp)
        m.fit(X_full, y_full, eval_set=[(X_tr, y_tr), (X_v, y_v)], eval_names=['train', 'val'])
    return m

def plot_loss(m, best_name, title):
    fig, ax = plt.subplots(figsize=(10, 5))
    if best_name == 'CatBoost':
        ev = m.get_evals_result()
        for ds, metrics in ev.items():
            for met, vals in metrics.items():
                ax.plot(vals, label=f'{ds} {met}')
    elif best_name == 'XGBoost':
        ev = m.evals_result()
        for ds, metrics in ev.items():
            for met, vals in metrics.items():
                ax.plot(vals, label=ds.replace('validation_0', 'train').replace('validation_1', 'val') + f' {met}')
    else:
        for ds, metrics in m.evals_result_.items():
            for met, vals in metrics.items():
                ax.plot(vals, label=f'{ds} {met}')
    ax.set_xlabel('Boosting Round'); ax.set_ylabel('Loss')
    ax.set_title(title); ax.legend(); ax.grid(True)
    plt.tight_layout(); plt.show()

print('Helpers ready.')

---
## 5. Baseline (Default Params)

Both datasets, default XGBoost/CatBoost/LightGBM, evaluated on validation set.

In [ ]:
default_models = {
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                              random_state=RANDOM_STATE, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                    random_state=RANDOM_STATE, verbose=0),
    'LightGBM': LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                                random_state=RANDOM_STATE, verbose=0),
}

base_rows = []
for tag, X_tr, X_v in [('Creation', Xc_train, Xc_val), ('Halfway', Xh_train, Xh_val)]:
    for name, m in default_models.items():
        r = evaluate_model(deepcopy(m), X_tr, X_v, y_train, y_val, f'{name} ({tag})')
        base_rows.append(r)

baseline_df = pd.DataFrame(base_rows)
baseline_df

---
## 6. Optuna Tuning — Creation-Time Dataset

In [ ]:
creation_models = {}
creation_optuna_rows = []
for mn in ['XGBoost', 'CatBoost', 'LightGBM']:
    print(f'\n=== Tuning {mn} (Creation) ===')
    m, best_val, bp = tune_one(mn, Xc_train, y_train)
    creation_models[mn] = m
    creation_optuna_rows.append({'model': mn, 'best_cv_roc_auc': best_val})

pd.DataFrame(creation_optuna_rows)

In [ ]:
# Validate tuned creation models
c_val_rows = []
for mn, m in creation_models.items():
    r = evaluate_model(deepcopy(m), Xc_train, Xc_val, y_train, y_val, mn)
    c_val_rows.append(r)
c_val_df = pd.DataFrame(c_val_rows)
best_c_name = c_val_df.sort_values('f1', ascending=False).iloc[0]['model']
print(f'Best creation model: {best_c_name}')
c_val_df

---
## 7. Optuna Tuning — Halfway Dataset

In [ ]:
halfway_models = {}
halfway_optuna_rows = []
for mn in ['XGBoost', 'CatBoost', 'LightGBM']:
    print(f'\n=== Tuning {mn} (Halfway) ===')
    m, best_val, bp = tune_one(mn, Xh_train, y_train)
    halfway_models[mn] = m
    halfway_optuna_rows.append({'model': mn, 'best_cv_roc_auc': best_val})

pd.DataFrame(halfway_optuna_rows)

In [ ]:
# Validate tuned halfway models
h_val_rows = []
for mn, m in halfway_models.items():
    r = evaluate_model(deepcopy(m), Xh_train, Xh_val, y_train, y_val, mn)
    h_val_rows.append(r)
h_val_df = pd.DataFrame(h_val_rows)
best_h_name = h_val_df.sort_values('f1', ascending=False).iloc[0]['model']
print(f'Best halfway model: {best_h_name}')
h_val_df

---
## 8. Fold-Safe CV (Both Datasets)

Quantify the leakage gap from pre-computed rate features.

In [ ]:
def fold_safe_cv(clf_class, X_full, X_fs, y, cv, kwargs):
    leaky, safe = [], []
    for tr_idx, va_idx in cv.split(X_full, y):
        m = clf_class(**kwargs); m.fit(X_full.iloc[tr_idx], y.iloc[tr_idx])
        leaky.append(roc_auc_score(y.iloc[va_idx], m.predict_proba(X_full.iloc[va_idx])[:, 1]))
        m = clf_class(**kwargs); m.fit(X_fs.iloc[tr_idx], y.iloc[tr_idx])
        safe.append(roc_auc_score(y.iloc[va_idx], m.predict_proba(X_fs.iloc[va_idx])[:, 1]))
    return np.array(leaky), np.array(safe)

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
specs = [
    ('XGBoost', XGBClassifier, {'random_state': RANDOM_STATE, 'eval_metric': 'logloss'}),
    ('CatBoost', CatBoostClassifier, {'random_seed': RANDOM_STATE, 'verbose': 0}),
    ('LightGBM', LGBMClassifier, {'random_state': RANDOM_STATE, 'verbose': 0}),
]

fold_rows = []
for tag, X_f, X_fs in [('Creation', Xc_train, Xc_train_fs), ('Halfway', Xh_train, Xh_train_fs)]:
    print(f'\n--- {tag} ---')
    for name, cc, kw in specs:
        l, s = fold_safe_cv(cc, X_f, X_fs, y_train, cv5, kw)
        fold_rows.append({'dataset': tag, 'model': name,
                          'leaky': f'{l.mean():.4f} +/- {l.std():.4f}',
                          'safe': f'{s.mean():.4f} +/- {s.std():.4f}',
                          'gap': f'{l.mean()-s.mean():.4f}'})
        print(f'  {name}: Leaky={l.mean():.4f}  Safe={s.mean():.4f}  Gap={l.mean()-s.mean():.4f}')

pd.DataFrame(fold_rows)

---
## 9. Final Models & Loss Curves

Retrain best creation and best halfway models on their respective **train+val** (fold-safe features).
Plot loss curves showing both training and validation loss.

In [ ]:
print('=== Final Creation Model ===')
final_c = train_final(creation_models[best_c_name], best_c_name,
                      Xc_train_fs, Xc_val_fs, y_train, y_val)
y_pred_c = final_c.predict(Xc_test_fs)
y_proba_c = final_c.predict_proba(Xc_test_fs)[:, 1]
plot_loss(final_c, best_c_name, f'Loss Curve — {best_c_name} (Creation)')

print('\n=== Final Halfway Model ===')
final_h = train_final(halfway_models[best_h_name], best_h_name,
                      Xh_train_fs, Xh_val_fs, y_train, y_val)
y_pred_h = final_h.predict(Xh_test_fs)
y_proba_h = final_h.predict_proba(Xh_test_fs)[:, 1]
plot_loss(final_h, best_h_name, f'Loss Curve — {best_h_name} (Halfway)')

---
## 10. Test Set Evaluation

In [ ]:
def show_eval(name, y_t, y_p, y_pp):
    print(f'\n=== {name} ===')
    cm = confusion_matrix(y_t, y_p)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Not Overdue', 'Overdue'],
                yticklabels=['Not Overdue', 'Overdue'])
    plt.title(f'{name} — Test Set'); plt.ylabel('Actual'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()
    print(classification_report(y_t, y_p, target_names=['Not Overdue', 'Overdue']))
    print(f'ROC-AUC: {roc_auc_score(y_t, y_pp):.4f}')
    pr, re, _ = precision_recall_curve(y_t, y_pp)
    print(f'PR-AUC: {auc(re, pr):.4f}')

show_eval(f'{best_c_name} (Creation)', y_test, y_pred_c, y_proba_c)
show_eval(f'{best_h_name} (Halfway)', y_test, y_pred_h, y_proba_h)

In [ ]:
# Side-by-side comparison
comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC'],
    f'Creation ({best_c_name})': [
        accuracy_score(y_test, y_pred_c),
        precision_score(y_test, y_pred_c, zero_division=0),
        recall_score(y_test, y_pred_c, zero_division=0),
        f1_score(y_test, y_pred_c, zero_division=0),
        roc_auc_score(y_test, y_proba_c),
        auc(*precision_recall_curve(y_test, y_proba_c)[:2]),
    ],
    f'Halfway ({best_h_name})': [
        accuracy_score(y_test, y_pred_h),
        precision_score(y_test, y_pred_h, zero_division=0),
        recall_score(y_test, y_pred_h, zero_division=0),
        f1_score(y_test, y_pred_h, zero_division=0),
        roc_auc_score(y_test, y_proba_h),
        auc(*precision_recall_curve(y_test, y_proba_h)[:2]),
    ],
}).set_index('Metric').round(4)
comparison

---
## Summary

| Aspect | Detail |
|--------|--------|
| Split | 70% train / 15% val / 15% test |
| Tracks | Creation-time + Halfway (independent tuning) |
| Tuning | Optuna, 50 trials × 3 models, 5-fold CV |
| Selection | Best model picked by val F1 per track |
| Fold-safe | 5 leaky features removed, gap quantified |
| Final | Trained on train+val, evaluated on test |
| Loss curves | Train + validation loss for both final models |